```sh
pip install sagemaker-core ipywidgets
```

### Notes

- Sagemaker container available support until python 3.12
- Referencia: https://github.com/aws/deep-learning-containers/blob/master/available_images.md

- Instance types to test: ml.g4dn.xlarge, ml.g6.xlarge, ml.trn1.2xlarge 
- https://github.com/aws/amazon-sagemaker-examples/blob/default/%20%20%20%20%20%20%20%20%20sagemaker-core/introductory-blog-sagemaker-core/introductory-blog-sagemaker-core.ipynb

In [ ]:
from datetime import datetime, timezone
from sagemaker_core.helper.session_helper import Session, s3_path_join

# Set up region, role and bucket parameters used throughout the notebook.
sagemaker_session = Session(
    default_bucket_prefix='congestion-bogota-training'
)
# region = sagemaker_session.boto_region_name
# role = get_execution_role()
region = 'us-east-1'
role = 'arn:aws:iam::195419001736:role/SagemakerExecutionRole'
bucket_name = sagemaker_session.default_bucket()

print(f"AWS region: {region}")
print(f"Execution role: {role}")
print(f"Default S3 bucket: {bucket_name}")

In [ ]:
import os
import tempfile
import tarfile

TAR_SOURCE_FILENAME = 'sourcedir.tar.gz'


def tar_and_upload_source_code(source_path: str, session: Session, s3_key_prefix: str='code'):
    '''Compress forder in .tar.gz format and upload to S3 project folder'''

    with tempfile.TemporaryDirectory() as tmp_dir_name:
        tar_file_name = '/'.join([tmp_dir_name, TAR_SOURCE_FILENAME])
        with tarfile.open(tar_file_name, mode='w:gz', dereference=True) as tar:
            for file_name in os.listdir(source_path):
                file_path = os.path.join(source_path, file_name)
                tar.add(file_path, arcname=os.path.basename(file_path))

        return session.upload_data(tar_file_name, key_prefix=s3_key_prefix)

source_code_uri = tar_and_upload_source_code('../src', sagemaker_session)

In [ ]:
image = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.7.1-cpu-py312-ubuntu22.04-sagemaker"
# image = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.7.1-gpu-py312-cu128-ubuntu22.04-sagemaker"

job_name = f"congestion-bogota-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"  # Name of training job
instance_type = "ml.t3.large"
# instance_type = "ml.g4dn.xlarge"  # SageMaker instance type to use for training

instance_count = 1  # Number of instances to use for training
volume_size_in_gb = 30  # Amount of storage to allocate to training job
max_runtime_in_seconds = 600  # Maximum runtime. Job exits if it doesn't finish before this
project_s3_path = s3_path_join("s3://", bucket_name, sagemaker_session.default_bucket_prefix)
s3_job_path = s3_path_join(project_s3_path, job_name)

train_data_uri = sagemaker_session.upload_data('../data/train', key_prefix='/data/train')
validation_data_uri = sagemaker_session.upload_data('../data/validation', key_prefix='/data/validation')
# train_data_uri = f'{s3_job_path}/data/train'
# validation_data_uri = f'{s3_job_path}/data/validation'
s3_output_path = project_s3_path

# Specify hyperparameters
hyper_parameters = {
    "epochs": "10",
    # "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    # "min_child_weight": "6",
    # "subsample": "0.8",
    # "verbosity": "0",
    # "objective": "binary:logistic",
    # "num_round": "100"

    "sagemaker_submit_directory": f'"{source_code_uri}"',
    'sagemaker_program': '"training_entrypoint.py"',
}

In [ ]:
from sagemaker_core.resources import TrainingJob
from sagemaker_core.shapes import (
    AlgorithmSpecification,
    Channel,
    DataSource,
    S3DataSource,
    ResourceConfig,
    StoppingCondition,
    OutputDataConfig,
    MetricDefinition
)

metric_definitions=[
    MetricDefinition(
        name="epoch",
        regex="epoch=([0-9]+)"
    ),
    MetricDefinition(
        name="train:loss",
        regex="train-loss=([0-9\\.]+)"
    ),
    MetricDefinition(
        name="train:accuracy",
        regex="train-accuracy=([0-9\\.]+)"
    )
]


# Create training job.
training_job = TrainingJob.create(
    training_job_name=job_name,
    hyper_parameters=hyper_parameters,
    algorithm_specification=AlgorithmSpecification(
        training_image=image,
        training_input_mode="File",
        metric_definitions=metric_definitions
    ),
    role_arn=role,
    region=region,
    input_data_config=[
        Channel(
            channel_name="train",
            content_type="csv",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=train_data_uri,
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
        Channel(
            channel_name="validation",
            content_type="csv",
            data_source=DataSource(
                s3_data_source=S3DataSource(
                    s3_data_type="S3Prefix",
                    s3_uri=validation_data_uri,
                    s3_data_distribution_type="FullyReplicated",
                )
            ),
        ),
    ],
    output_data_config=OutputDataConfig(s3_output_path=s3_output_path),
    resource_config=ResourceConfig(
        instance_type=instance_type,
        instance_count=instance_count,
        volume_size_in_gb=volume_size_in_gb,
        # https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html
        # keep_alive_period_in_seconds=60
    ),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=max_runtime_in_seconds),
    # environment={
    #     "PIP_CACHE_DIR": "/opt/ml/sagemaker/warmpoolcache/pip",
    # }
)

# Wait for the training job to complete
# training_job.wait()
training_job.wait(poll=20, timeout=None, logs=False)